In [ ]:
import torch, importlib, gc, os, sys, json, datetime, random, warnings
print("torch", torch.__version__, "| cuda? →", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/absolute/path/to/')

In [ ]:
!git clone -q https://github.com/yingtaoluo/Spatial-Temporal-Attention-Network-for-POI-Recommendation.git
%cd Spatial-Temporal-Attention-Network-for-POI-Recommendation

In [ ]:
# !pip install "numpy<2.0"

In [ ]:
import pandas as pd
import numpy as np
import pickle as pkl
import json
import json, glob
from tqdm import tqdm
import os
import importlib, site, sys
from tqdm.auto import tqdm

In [ ]:
base_path =  '/absolute/path/to/STAN'
original_path = '/absolute/path/to/SafetyIsAllYouNeed/baselines/exports'

### Load original trajectories

In [ ]:
with open(os.path.join(original_path, 'train_trajectories.pickle'), 'rb') as f:
    train_trajs = pkl.load(f)

with open(os.path.join(original_path, 'validation_trajectories.pickle'), 'rb') as f:
    validation_trajs = pkl.load(f)

with open(os.path.join(original_path, 'test_trajectories.pickle'), 'rb') as f:
    test_trajs = pkl.load(f)

train_df = pd.read_csv(os.path.join(original_path, 'train_checkins.csv'))
validation_df = pd.read_csv(os.path.join(original_path, 'validation_checkins.csv'))
test_df = pd.read_csv(os.path.join(original_path, 'test_checkins.csv'))

### Convert trajectories → STAN raw format


In [ ]:
def dump_to_txt(trajectories, out_path):
    """
    STAN expects *one check-in per line* in chronological order:
      user poi cat lat lon utc_ts
    separated with TABs.  (Anything after the 6-th column is ignored
    by their dataloader, so we skip timezone, etc.)
    """
    rows = []
    for df in trajectories:
        df = df.sort_values("utc_time")                       # strict chrono
        for _, r in df.iterrows():
            rows.append([
                int(r.user_id),
                int(r.poi_id),
                str(r.poi_category_id),
                f"{r.latitude:.6f}",
                f"{r.longitude:.6f}",
                int(pd.Timestamp(r.utc_time).timestamp())     # seconds UTC
            ])
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        for row in rows:
            f.write("\t".join(map(str,row))+"\n")

RAW_DIR = "data/nyc/raw_len20"          # keep everything inside repo
dump_to_txt(train_trajs, f"{RAW_DIR}/train.txt")
dump_to_txt(validation_trajs,   f"{RAW_DIR}/val.txt")
dump_to_txt(test_trajs,  f"{RAW_DIR}/test.txt")
print("✔️  Raw txt files ready")


### Create a minimal config

In [ ]:
import json, textwrap, pathlib, pprint, os
conf = {
    "dataset":      "nyc_len20",           # just a tag; any string
    "data_dir":     RAW_DIR,               # <- path we just created
    "emb_size":     128,
    "hidden_size":  256,
    "epochs":       20,
    "batch_size":   64,
    "lr":           1e-3,
    "history_len":  19,          # <<== first 19 check-ins are the context
    "predict_k":    1,           # predict the very next check-in
    "cuda":         True,        # will fallback to CPU if runtime has no GPU
    "topk_eval":    [1,3,5]      # we’ll inject Acc@1/3/5 evaluation
}
os.makedirs("conf", exist_ok=True)
with open("conf/nyc_len20.json", "w") as f: json.dump(conf, f, indent=2)
pprint.pprint(conf)

### Train

In [ ]:
from train import main as stan_main
import json, argparse, torch

cfg = argparse.Namespace(**json.load(open("conf/nyc_len20.json")))
if not torch.cuda.is_available(): cfg.cuda = False
stan_main(cfg)


from data_utils import build_data
from model import STAN
import metrics, torch.nn.functional as F, numpy as np, torch

# 1.  build vocab & loaders exactly like train.py
train_loader, valid_loader, test_loader, n_user, n_loc, n_cat = build_data(cfg)

# 2.  load best checkpoint (train.py saved it to ckpt/best_model.pth)
best_state = torch.load("ckpt/best_model.pth", map_location="cpu")
model = STAN(n_user, n_loc, n_cat, cfg).to(cfg.device if cfg.cuda else "cpu")
model.load_state_dict(best_state["model"])

# 3.  evaluate on test split
results = metrics.evaluate(model, test_loader,
                           k_list=[1,3,5],        # we only care about 1/3/5
                           device=cfg.device if cfg.cuda else "cpu")

print("\n=== FINAL TEST METRICS ===")
for k in (1,3,5):
    print(f"Acc@{k}: {results[f'Acc@{k}']:.4f}")
print(f"MRR   : {results['MRR']:.4f}")
